In [0]:
len(dbutils.fs.ls('/Workspace/Users/jaquesada92@outlook.com/contraloria_panama/staging/'))

In [0]:
len(dbutils.fs.ls('/Volumes/contraloria/reference_and_audit/contraloria_staging'))

In [0]:
import os
import shutil

for filename in os.listdir(os.path.dirname(src)):
    if filename.endswith('.parquet'):
        shutil.copy2(
            os.path.join(os.path.dirname(src), filename),
            os.path.join(dst, filename)
        )

In [0]:
import os

staging_dir = '/Workspace/Users/jaquesada92@outlook.com/contraloria_panama/staging/'

# List all files in the staging directory
try:
    files = os.listdir(staging_dir)
    print(f"Total files in staging directory: {len(files)}\n")
    
    # Show all files
    for f in sorted(files):
        file_path = os.path.join(staging_dir, f)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path)
            print(f"{f} - Size: {size:,} bytes")
        else:
            print(f"{f} - (directory)")
            
    # Check specifically for the problematic file
    problematic_file = "InformeConsultaPlanilla_02042026_ASAMBLEA_NACIONAL_EVENTUAL_1775031625_0.parquet"
    if problematic_file in files:
        print(f"\n✓ The file '{problematic_file}' EXISTS in the directory")
        full_path = os.path.join(staging_dir, problematic_file)
        print(f"  Size: {os.path.getsize(full_path):,} bytes")
    else:
        print(f"\n✗ The file '{problematic_file}' DOES NOT EXIST in the directory")
        
except Exception as e:
    print(f"Error listing directory: {e}")

In [0]:
spark.read.parquet("dbfs:/Workspace/Users/jaquesada92@outlook.com/contraloria_panama/staging/InformeConsultaPlanilla_02042026_ASAMBLEA_NACIONAL_EVENTUAL_1775031625_0.parquet").display()

In [0]:
spark.read.parquet('dbfs:/Workspace/Users/jaquesada92@outlook.com/contraloria_panama/staging/InformeConsultaPlanilla_02042026_ASAMBLEA_NACIONAL_EVENTUAL_1775031625_0.parquet').display()

In [0]:
# Solución 1: Leer todos los archivos de ASAMBLEA_NACIONAL juntos
df_asamblea = spark.read.parquet('/Workspace/Users/jaquesada92@outlook.com/contraloria_panama/staging/InformeConsultaPlanilla_02042026_ASAMBLEA_NACIONAL_*.parquet')
display(df_asamblea)

In [0]:
# Solución 2: Copiar el archivo a DBFS temporal y leerlo desde allí
import shutil
import os

# Ruta origen
source_path = '/Workspace/Users/jaquesada92@outlook.com/contraloria_panama/staging/InformeConsultaPlanilla_02042026_ASAMBLEA_NACIONAL_EVENTUAL_1775031625_0.parquet'

# Ruta destino en DBFS
dest_path = '/tmp/temp_asamblea_eventual.parquet'

try:
    # Copiar archivo
    shutil.copy2(source_path, dest_path)
    print(f"✓ Archivo copiado exitosamente a {dest_path}")
    
    # Intentar leer desde la nueva ubicación
    df_temp = spark.read.parquet(dest_path)
    print(f"\n✓ DataFrame cargado exitosamente")
    print(f"  Filas: {df_temp.count()}")
    print(f"  Columnas: {len(df_temp.columns)}")
    
    display(df_temp)
    
except Exception as e:
    print(f"✗ Error: {e}")

In [0]:
# Solución 3: Leer con pandas y convertir a Spark
import pandas as pd
import pyarrow.parquet as pq

# Ruta del archivo problemático
file_path = '/Workspace/Users/jaquesada92@outlook.com/contraloria_panama/staging/InformeConsultaPlanilla_02042026_ASAMBLEA_NACIONAL_EVENTUAL_1775031625_0.parquet'

try:
    # Leer con pandas
    pdf = pd.read_parquet(file_path)
    print(f"✓ Archivo leído con pandas")
    print(f"  Filas: {len(pdf)}")
    print(f"  Columnas: {len(pdf.columns)}")
    print(f"\nColumnas: {list(pdf.columns)}")
    
    # Convertir a Spark DataFrame
    df_asamblea_eventual = spark.createDataFrame(pdf)
    print(f"\n✓ Convertido a Spark DataFrame")
    
    display(df_asamblea_eventual)
    
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

## ¿Por qué Spark no puede leer archivos del Workspace?

El **Workspace filesystem** (`/Workspace/...`) está diseñado para notebooks y archivos pequeños, **NO para datos que Spark procesa**.

## Solución recomendada: Mover archivos a Unity Catalog Volume

Los **Volumes** son el lugar correcto para almacenar datos que Spark necesita leer:

```python
# 1. Crear un volume (una sola vez)
spark.sql("""
  CREATE VOLUME IF NOT EXISTS main.default.contraloria_staging
""")

# 2. Copiar archivos del workspace al volume
import shutil
import os

source_dir = '/Workspace/Users/jaquesada92@outlook.com/contraloria_panama/staging/'
dest_dir = '/Volumes/main/default/contraloria_staging/'

# Copiar todos los archivos parquet
for filename in os.listdir(source_dir):
    if filename.endswith('.parquet'):
        shutil.copy2(
            os.path.join(source_dir, filename),
            os.path.join(dest_dir, filename)
        )
        print(f"Copiado: {filename}")

# 3. Ahora Spark PUEDE leer desde el Volume
df = spark.read.parquet('/Volumes/main/default/contraloria_staging/*.parquet')
print(f"Total filas: {df.count()}")
```

## Para tu pipeline:

Cambia la ruta en tu pipeline de:
- ❌ `/Workspace/Users/.../staging/...parquet`

A:
- ✅ `/Volumes/main/default/contraloria_staging/*.parquet`

In [0]:
import shutil
import os

# 1. Crear el volume (si no existe)
print("Creando volume...")
try:
    spark.sql("""
      CREATE VOLUME IF NOT EXISTS main.default.contraloria_staging
    """)
    print("✓ Volume creado/verificado: main.default.contraloria_staging")
except Exception as e:
    print(f"✗ Error creando volume: {e}")
    print("Nota: Si no tienes permisos, pide al admin que cree el volume")

# 2. Copiar archivos
source_dir = '/Workspace/Users/jaquesada92@outlook.com/contraloria_panama/staging/'
dest_dir = '/Volumes/main/default/contraloria_staging/'

print(f"\nCopiando archivos de:")
print(f"  Origen: {source_dir}")
print(f"  Destino: {dest_dir}\n")

files_copied = 0
errors = 0

for filename in os.listdir(source_dir):
    if filename.endswith('.parquet'):
        try:
            source_file = os.path.join(source_dir, filename)
            dest_file = os.path.join(dest_dir, filename)
            shutil.copy2(source_file, dest_file)
            files_copied += 1
            if files_copied <= 5:  # Mostrar solo los primeros 5
                print(f"✓ {filename}")
        except Exception as e:
            errors += 1
            print(f"✗ Error con {filename}: {e}")

if files_copied > 5:
    print(f"  ... y {files_copied - 5} archivos más")

print(f"\nResumen:")
print(f"  Archivos copiados: {files_copied}")
print(f"  Errores: {errors}")

# 3. Verificar que Spark puede leer
if files_copied > 0:
    print("\nVerificando lectura con Spark...")
    try:
        df_volume = spark.read.parquet('/Volumes/main/default/contraloria_staging/*.parquet')
        print(f"✓ Spark puede leer los archivos")
        print(f"  Total filas: {df_volume.count():,}")
        print(f"  Columnas: {len(df_volume.columns)}")
        display(df_volume.limit(5))
    except Exception as e:
        print(f"✗ Error leyendo con Spark: {e}")